# Kütüphaneler

In [63]:
import pandas as pd
import numpy as np
import pandas.io.formats.excel as pife
import warnings
import pickle
import os
import locale
from termcolor import colored
from pandas.errors import SettingWithCopyWarning
from datetime import datetime

# Ayarlar

In [125]:
target_end_term = '2023.12'
pra_v = 'v12.2'
# output_extra = False
save_output = True

scope = 1
target_start_term = None
skip_last_month = False
os.system('color')
locale.setlocale(locale.LC_ALL, 'tr_TR')
pd.options.display.float_format = '{:,.2f}'.format
pife.ExcelFormatter.header_style = None

warnings.filterwarnings('ignore', category = UserWarning)
warnings.filterwarnings('ignore', category = pd.errors.ParserWarning)
warnings.filterwarnings('ignore', category = SettingWithCopyWarning)

base_path = 'C:\\Users\\105617\\Desktop\\Workspace\\Panel\\'

pickle_folder_path = base_path + 'pickle' + '\\'
data_folder_path = base_path + 'data' + '\\'
output_folder_path = base_path + 'output' + '\\'

kr202_folder_path = data_folder_path + 'KR202' + '\\'
ms_folder_path = data_folder_path + 'Müşteri Sınıfı' + '\\'
proje_folder_path = data_folder_path + 'Proje Kredileri' + '\\'
memzuc_folder_path = data_folder_path + 'Memzuç' + '\\'

yz_folder_path = data_folder_path + 'İzleme' + '\\'
eds_folder_path = data_folder_path + 'Entegre Derece Skor' + '\\'
fg_folder_path = data_folder_path + 'Finansal Güçlük' + '\\'
kroa_folder_path = data_folder_path + 'KRÖA' + '\\'
lgd_folder_path = data_folder_path + 'LGD' + '\\'

mali_folder_path = data_folder_path + 'Mali Tablolar' + '\\'
kur_folder_path = data_folder_path + 'Kur Riskliliği' + '\\'
sektor_folder_path = data_folder_path + 'Sektör Dereceleri' + '\\'

for p in [
    data_folder_path, output_folder_path, pickle_folder_path,
    kr202_folder_path, ms_folder_path, proje_folder_path, memzuc_folder_path,
    yz_folder_path, fg_folder_path, kroa_folder_path, lgd_folder_path,
    mali_folder_path, kur_folder_path, sektor_folder_path,
]:
    os.makedirs(p, exist_ok = True)

# Yardımcı Fonksiyonlar

## Tablo Manipülasyonu

In [126]:
def add_h_space(dfo, space=1):
    df = dfo.copy()
    w = df.shape[1]
    for _ in range(space):
        df.loc[len(df)] = [np.nan] * w
    return df

def add_v_space(dfo, space=1):
    df = dfo.copy()
    return pd.concat([df, pd.DataFrame([[np.nan] * space], columns=[np.nan] * space)], axis=1)
    

def h_stack(dfo, space=3):
    df = dfo.copy()
    df = df[0].reset_index(drop=True)
    dfln = [df]
    dfna = pd.DataFrame({np.nan: [np.nan]})
    for dftt in dfo[1:]:
        dft = dftt.copy()
        for i in range(space):
            dfln.append(dfna)
        dfln.append(dft.reset_index(drop=True))
    df = pd.concat(dfln, axis=1).reset_index(drop=True)
    return df


def v_stack(dfo, space=3):
    df = dfo.copy()
    df = df[0].reset_index(drop=True)
    dfln = [df]
    dfna = pd.DataFrame(columns=df.columns)

    dfna = add_h_space(dfna, space)
    for dftt in dfo[1:]:
        dft = dftt.copy()
        l = df.shape[1]
        lt = dft.shape[1]
        diff = l - lt

        if diff > 0:
            dft = add_v_space(dft, diff)
        elif diff < 0:
            df = add_v_space(df, -diff)

        dfc = pd.DataFrame([dft.columns], columns=df.columns)
        dft.columns = df.columns
        dfln.append(dfna)
        dfln.append(dfc)
        dfln.append(dft.reset_index(drop=True))

    df = pd.concat(dfln).reset_index(drop=True)
    return df


def insert_header(dfo, col):
    df = dfo.copy()
    header = [col] if type(col) is str else col
    return v_stack([pd.DataFrame(columns=header), df], 0)

def insert_corner(dfo):
    df = dfo.copy()
    return h_stack([pd.DataFrame(), v_stack([pd.DataFrame(), df], 0)], 1)

## Verim Tabloları

In [127]:
def fix_verim_table(df, columns=None):
    start = 0
    if columns is None:
        start = 3
        columns = df.iloc[2, 1:]
    df = df.iloc[start:, 1:]
    df.columns = columns
    df.columns.name = None
    df = df.reset_index(drop=True)
    return df

def quick_export_verim_csv(data, output_file_name, sep=';', header=False, index=False, open_file=False):
    cl = [str(x) for x in range(0, 16)]
    df = pd.DataFrame()
    df['10'] = data
    for c in ['0', '15']:
        df[c] = 'x'
    for c in cl:
        if c not in ['0', '10', '15']:
            df[c] = np.nan
    df = df[cl]

    output_file_path = output_folder_path + output_file_name + '.csv'
    df.to_csv(output_file_path, sep=sep, header=header, index=index)

    if open_file:
        os.startfile(output_file_path)
    

## Hızlı Çıktı

In [128]:
def quick_export(df_export, output_file_name, sheet_name=None, open_file=True):
    output_file_path = output_folder_path + output_file_name + '.xlsx'
    writer = pd.ExcelWriter(output_file_path, engine = 'xlsxwriter')
    if type(df_export) is list:
        if sheet_name is None:
            sheet_name = [x + 1 for x in range(len(df_export))]
        for df, sheet in zip(df_export, sheet_name):
            df.to_excel(writer, sheet_name = sheet, index = False)
    else:
        if sheet_name is None:
            sheet_name = output_file_name
        df_export.to_excel(writer, sheet_name = sheet_name, index = False)
    writer.close()
    if open_file:
        os.startfile(output_file_path)


def quick_export_csv(df, file_name, open_file=False):
    df.to_csv(output_folder_path + file_name  + '.csv', sep=';', encoding='ANSI', index=False)

## Now

In [129]:
def now():
    return datetime.strftime(datetime.now(), '%m%d%H%M')

## Pickle

In [130]:
def save_pickle(file, name, sub=''):
    os.makedirs(pickle_folder_path + sub, exist_ok = True)
    with open(pickle_folder_path + sub + name, 'wb') as handle:
        pickle.dump(file, handle, protocol=pickle.HIGHEST_PROTOCOL)

def load_pickle(name, sub=''):
    with open(pickle_folder_path + sub + name, 'rb') as handle:
        return pickle.load(handle)

## Toplu Dosya İşleme

In [131]:
def scan_folder(folder_path, extension='.xlsx', offset=0):
    file_list, date_list = [], []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(extension) and not file_name.startswith('~$'):
            s = file_name.split('.')
            y = s[0][-4:]
            m = s[1]
            d = s[2][:2]
            file_list.append(file_name)
            date_list.append(y + '.' + m + '.' + d)

    sorted_list = sorted(zip(date_list, file_list))

    if target_end_term is not None:
        sorted_list = sorted_list[:[x[0][:7] for x in sorted_list].index(target_end_term) + 1 + offset]
    elif skip_last_month:
        sorted_list = sorted_list[:-1]

    if target_start_term is not None:
        sorted_list = sorted_list[[x[0][:7] for x in sorted_list].index(target_end_term) + offset:]
    else:
        sorted_list = sorted_list[-scope:]

    return file_list.copy(), date_list.copy(), sorted_list.copy()

In [132]:
def read_file(data_folder_path, file_name, fix_verim=False, csv=None, target_sheet_list=None):
    print(colored(file_name, 'light_green'), 'okunuyor')
    if csv is not None:
        sep, encoding, low_memory, index_col = ';', 'ANSI', False, True
        if type(csv) is list:
            try:
                sep = csv[0]
                encoding = csv[1]
                low_memory = csv[2]
                index_col = csv[3]
            except:
                pass
        if index_col:
            df = pd.read_csv(data_folder_path + file_name, sep=sep, encoding=encoding, low_memory=low_memory)
        else:
            df = pd.read_csv(data_folder_path + file_name, sep=sep, encoding=encoding, low_memory=low_memory, index_col=index_col)
        print(colored(file_name, 'light_green'), 'okundu.')

    else:
        xls = pd.ExcelFile(data_folder_path + file_name)
        df = pd.DataFrame()
        sheet_list = target_sheet_list if target_sheet_list is not None else xls.sheet_names
        for sheet in sheet_list:
            print('\t', colored(sheet, 'light_cyan'), 'okunuyor')
            dfs = pd.read_excel(xls, sheet_name=sheet)
            if fix_verim:
                dfs = fix_verim_table(dfs)
            df = pd.concat([df, dfs], ignore_index=True) if len(sheet_list) > 1 else dfs
        xls.close()
        print(colored(file_name, 'light_green'), 'okundu.')
            
    return df.copy()

In [133]:
def batch_import_files(sorted_list, data_folder_path, pickle_sub, pickle_name_base, fix_verim=False, csv=None, target_sheet_list=None):
    sorted_date_list, df_raw_list = [], []
    pickle_sub += '\\'
    pickle_name_base += '_'
    l = len(sorted_list)
    for i, (date, file_name) in enumerate(sorted_list):
        sorted_date_list.append(date[:7])
        pickle_name = pickle_name_base + date
        ps = f'({round((i + 1) / l * 100)}%)'
        ps = ' ' * (len('(100%)') - len(ps)) + ps
        if os.path.exists(pickle_folder_path + pickle_sub + pickle_name):
            df = load_pickle(pickle_name, pickle_sub)
            print(colored(file_name, 'light_green'), 'önbellekten okundu', colored(ps, 'light_green'))
        else:
            df = read_file(data_folder_path, file_name, fix_verim, csv, target_sheet_list)
            if csv is None:
                print(colored(file_name, 'light_green'), 'kaydediliyor')
                save_pickle(df, pickle_name, pickle_sub)
                print(colored(file_name, 'light_green'), 'kaydedildi.', colored(ps, 'light_green'))

        df_raw_list.append(df.copy())
    print('Tüm dosyalar başarıyla okundu')

    return sorted_date_list.copy(), df_raw_list.copy()

In [134]:
def batch_process_files(df_raw_list, sorted_list, sorted_date_list, fix_df_function, column_list=None, fill_string=None, rename_exclude_columns = 'Müşteri No'):
    df_all = []
    df = pd.DataFrame()
    l = len(df_raw_list)
    for i, dfi in enumerate(df_raw_list):
        df = dfi.copy()
        
        if fix_df_function is not None:
            df = fix_df_function(df, i)

        if i == len(df_raw_list) - 1:
            df_last = df.copy().reset_index(drop=True)

        if column_list is not None:  
            df = df[column_list]
     
        if type(rename_exclude_columns) is str:
            df.columns = [c + ' ' + sorted_date_list[i] if c != rename_exclude_columns else c for c in df.columns]
        else:
            df.columns = [c + ' ' + sorted_date_list[i] if c not in rename_exclude_columns else c for c in df.columns]
        df = df.sort_values('Müşteri No').reset_index(drop=True)

        if fill_string is not None:
            df = df.fillna(fill_string)

        df_all.append(df)        
        ps = f'({round((i + 1) / l * 100)}%)'
        ps = ' ' * (len('(100%)') - len(ps)) + ps
        print(colored(sorted_list[i][1], 'light_green'), 'işlendi', colored(ps, 'light_green'))

    print('Tüm dosyalar başarıyla oluşturuldu')

    return df_all.copy(), df_last.copy()

In [135]:
def get_recent_file(data_folder_path, pickle=None, fix_verim=False, csv=None, target_sheet_list=None):
    file_list = []
    for file_name in os.listdir(data_folder_path):
        file_list.append(file_name)

    if target_end_term is None:
        target_file_name = file_list[-1]
    else:
        target_end_date = int(target_end_term[:4]) * 12 + int(target_end_term[-2:])
        date_list = [x.split(' - ')[1][:7] for x in file_list]
        date_list = [int(x[:4]) * 12 + int(x[-2:]) - target_end_date for x in date_list]
        target_index = date_list.index(max([x for x in date_list if x <= 0]))
        target_file_name = file_list[target_index]

    if pickle is None:
        df = read_file(data_folder_path, target_file_name, fix_verim, csv, target_sheet_list)
    else:
        pickle_name, pickle_sub = pickle[0], pickle[1]
        if os.path.exists(pickle_folder_path + pickle_sub + pickle_name):
            df = load_pickle(pickle_name, pickle_sub)
            print(colored(file_name, 'light_green'), 'önbellekten okundu')
        else:
            df = read_file(data_folder_path, file_name, fix_verim, csv, target_sheet_list)
            print(colored(file_name, 'light_green'), 'kaydediliyor')
            save_pickle(df, pickle_name, pickle_sub)
            print(colored(file_name, 'light_green'), 'kaydedildi.')

    return df.copy()

# Tekil Dosyalar

## Mali Tablolar

In [136]:
mali_columns = [
    'CST_NMR', 'FST_YR', 'FST_MO', 'DONEM', 'AKTIF', 'DONEN_DEGERLER', 'PARA_MEVCUDU', 'MENKUL_DEGERLER',
    'ZARAR', 'KISA_VADELI_BORCLAR', 'KV_FINANSAL_BORCLAR', 'UZUN_VADELI_BORCLAR', 'UV_FINANSAL_BORCLAR',
    'OZ_SERMAYE', 'KAR', 'NET_SATISLAR', 'FAALIYET_GELIRI', 'FINANSMAN_GIDERLERI', 'VFO_KAR',
    'DONEM_AMORTISMAN', 'TOP_FIN_BORC', 'TOP_BORCLAR', 'YENI_OZSERMAYE', 'EBITDA',
    'KVFB_NET_SATISLAR', 'BORCLANMA_ORANI', 'KALDIRAC_ORANI', 'FAIZ_YUKU_KARSILAMA'
]

mali_float_columns = [x for x in mali_columns if x not in ['SNPST_DT', 'DONEM', 'BILANCO_GIRISI_TAMAMLANDI', 'BANK_FORMAT_TAMAMLANDI']]

mali_column_rename_dict = {
    'CST_NMR': 'Müşteri No',
    'KVFB_NET_SATISLAR': 'KVFB / Net Satışlar',
    'BORCLANMA_ORANI': 'Borçlanma Oranı',
    'KALDIRAC_ORANI': 'Kaldıraç Oranı',
    'FAIZ_YUKU_KARSILAMA': 'Faiz Yükü Karşılama Farkı',
}

def mali_cari_oran(df):
    if df['KISA_VADELI_BORCLAR'] == 0:
        return np.nan
    else:
        return df['DONEN_DEGERLER'] / df['KISA_VADELI_BORCLAR']

def fix_mali_df(df):
    df = df.loc[df['RN'] == 1, mali_columns]
    
    for c in mali_float_columns:
        if df[c].dtype == 'O':
            df[c] = df[c].apply(lambda x: float(x.replace(',', '.')) if pd.notnull(x) else np.nan)

    df = df.rename(columns=mali_column_rename_dict)
    df['Cari Oran'] = df.apply(mali_cari_oran, axis=1)
    return df.copy()

df_mali = get_recent_file(mali_folder_path, csv=[';'])
df_mali = fix_mali_df(df_mali)

MT - 2023.12.01.csv okunuyor
MT - 2023.12.01.csv okundu.


## Kur Riski Çalışması

In [137]:
df_kur = get_recent_file(kur_folder_path, csv=[';'])

Kur Riskliliği - 2023.12.31.csv okunuyor
Kur Riskliliği - 2023.12.31.csv okundu.


## Sektör Risklilik Çalışması

In [138]:
sektor_column_list = ['NACE REV.2 KODU', 'EPP', 'İZLEME']
sektor_renamed_column_list = ['NACE 6lı', 'Sektör Derecesi', 'Sektör Derece İzleme']

def fix_sektor_df(df):
    df = df[sektor_column_list]
    df.columns = sektor_renamed_column_list
    df = df.loc[df['NACE 6lı'].apply(lambda x: len(x) == 8)].reset_index(drop=True)
    return df.copy()

df_sektor = get_recent_file(sektor_folder_path)
df_sektor = fix_sektor_df(df_sektor)

Sektör - 2023.07.01.xlsx okunuyor
	 EPD Sonuçları - Temmuz 2023 okunuyor
Sektör - 2023.07.01.xlsx okundu.


## Şube Bilgileri

In [139]:
# if os.path.exists(pickle_folder_path + 'df_sube'):
#     df_sube = load_pickle('df_sube')
# else:
#     xls = pd.ExcelFile(data_folder_path + sube_file_name)
#     df_sube = pd.read_excel(xls)
#     save_pickle(df_sube, 'df_sube')
#     xls.close()

## Müşteri No - VKN - TCKN

In [140]:
# if os.path.exists(pickle_folder_path + 'df_mnvt'):
#     df_mnvt = load_pickle('df_mnvt')
# else:
#     df_mnvt = pd.read_csv(data_folder_path + mnvt_file_name, sep=';')
#     save_pickle(df_mnvt, 'df_mnvt')

# df_mnvt = df_mnvt.loc[(df_mnvt['TCKN'].notnull()) | (df_mnvt['VKN'].notnull())]

# KRÖA'ya Yaklaşma

In [141]:
if target_end_term is not None and target_end_term.startswith('2022'):
    df_kroa_yaklasma = pd.read_csv(data_folder_path + 'KRÖA Yaklaşma - 2022.11.01.csv', sep=';', encoding='ANSI')
else:
    df_kroa_yaklasma = pd.read_csv(data_folder_path + 'KRÖA Yaklaşma - 2023.11.01.csv', sep=';', encoding='ANSI')

# İntikal Öngörüsü

In [142]:
if target_end_term is not None and target_end_term.startswith('2022'):
    df_ongoru = pd.DataFrame(columns=['Müşteri No', '2024 Öngörüsü'])
else:
    df_ongoru = pd.read_csv(data_folder_path + '2024 Öngörüsü.csv', sep=';', encoding='ANSI')

# Veri İçe Alma

## KR202

In [143]:
kr202_file_list, kr202_date_list, kr202_sorted_list = scan_folder(kr202_folder_path)
kr202_sorted_date_list, kr202_df_raw_list = batch_import_files(kr202_sorted_list, kr202_folder_path, 'KR202', 'kr202')

last_date = kr202_sorted_list[-1][0][:7]

KR202 - 2023.12.31.xlsx önbellekten okundu (100%)
Tüm dosyalar başarıyla okundu


In [144]:
kr202_column_list = [
    'Müşteri No',
    'Şube Kodu',
    'Şube Adı',
    'Şube İl',
    'Şube İlçe',
    'Şube Türü',
    'Bölge Adı',
    'İlgili Bölüm',
    'Yetki Kodu',
    'NACE Kodu',
    'NACE Harf',
    'Tahsis Sektör Adı',
    'Nakdi TL Reeskontlu',
    'Nakdi YP Reeskontlu',
    'Nakdi Reeskontlu Toplam',
    'G.Nakdi TL Bakiye',
    'G.Nakdi YP Bakiye',
    'G.Nakdi Toplam',
    'Toplam Reeskontlu Risk',
    'Nakdi Karşılık',
    'G.Nakdi Karşılık',
    'Toplam Karşılık',
    'Nakdi Karşılık Oranı',
    'G.Nakdi Karşılık Oranı',
    'Karşılık Oranı',
]

kr202_initial_exclude_list = [
    'NACE Harf',
    'Nakdi Karşılık Oranı',
    'G.Nakdi Karşılık Oranı',
    'Karşılık Oranı',
]

kr202_initial_column_list = [x for x in kr202_column_list if x not in kr202_initial_exclude_list]

kr202_include_list = [
    'Müşteri No', 'Adı', 'Şube Kodu', 'Şube Adı', 'Şube İl', 'Şube İlçe',
    'Şube Türü', 'Bölge Adı', 'Yetki Kodu', 'İlgili Bölüm', 'Risk Grubu Kodu', 'Risk Grubu Adı',
    'NACE Kodu', 'Tahsis Sektör Adı', 'Tahsis Alt Sektör Adı',
    'Nakdi TL Bakiye', 'Nakdi YP Bakiye', 'Nakdi Toplam',
    'G.Nakdi TL Bakiye', 'G.Nakdi YP Bakiye', 'G.Nakdi Toplam', 'Toplam Risk',
    'Nakdi TL Reeskontlu', 'Nakdi YP Reeskontlu', 'Nakdi Reeskontlu Toplam', 'Toplam Reeskontlu Risk',
    'Nakdi Karşılık', 'G.Nakdi Karşılık', 'Toplam Karşılık', 
]

kr202_float_list = [
    'Müşteri No', 'Şube Kodu', 'Yetki Kodu',
    'Nakdi TL Bakiye', 'Nakdi YP Bakiye', 'Nakdi Toplam',
    'G.Nakdi TL Bakiye', 'G.Nakdi YP Bakiye', 'G.Nakdi Toplam', 'Toplam Risk',
    'Nakdi TL Reeskontlu', 'Nakdi YP Reeskontlu', 'Nakdi Reeskontlu Toplam', 'Toplam Reeskontlu Risk',
    'Nakdi Karşılık', 'G.Nakdi Karşılık', 'Toplam Karşılık',
]

kr202_column_rename_dict = {
	'MUSTERI_NO': 'Müşteri No',
    'Musteri No': 'Müşteri No',
	'ADI': 'Adı',
    'Adi': 'Adı',
	'SANTRAL_SUBE_KODU': 'Şube Kodu',
    'SANTRAL SUBE KODU': 'Şube Kodu',
    'Santral Şube Kodu': 'Şube Kodu',
	'SANTRAL_SUBE_ADI': 'Şube Adı',
    'Santral Şube Adı': 'Şube Adı',
	'SANTRAL_SUBE_ILI': 'Şube İl',
    'SANTRAL_SUBE_IL': 'Şube İl',
    'SANTRAL ŞUBE İL': 'Şube İl',
    'Santral  Şube İl': 'Şube İl',
	'SANTRAL_SUBE_ILCE': 'Şube İlçe',
    'SANTRAL_SUBE_ILCE': 'Şube İlçe',
    'SANTRAL ŞUBE İLÇE': 'Şube İlçe',
    'Santral Şube İlçe': 'Şube İlçe',
	'SANTRAL_SUBE_TURU': 'Şube Türü',
    'Santral Şube Türü': 'Şube Türü',
	'BOLGE_ADI': 'Bölge Adı',
	'SANRAL_BOLGE_ADI': 'Bölge Adı',
	'YETKI_KODU': 'Yetki Kodu',
	'BOLUM_ADI': 'İlgili Bölüm',
	'RISK_GRUBU_KODU': 'Risk Grubu Kodu',
	'RISK_GRUBU_ADI': 'Risk Grubu Adı',
	'NACE_KODU': 'NACE Kodu',
    'Nace Kodu': 'NACE Kodu',
	'TAHSIS_SEKTOR_ADI': 'Tahsis Sektör Adı',
    'SEKTÖR': 'Tahsis Sektör Adı',
	'TAHSIS_ALT_SEKTOR_ADI': 'Tahsis Alt Sektör Adı',
    'Tahsis Alt Sektör AdI': 'Tahsis Alt Sektör Adı',
    'ALT SEKTÖR': 'Tahsis Alt Sektör Adı',
    'NAKDI_TL_BAK': 'Nakdi TL Bakiye',
	'NAKDI_YP_BAK': 'Nakdi YP Bakiye',
	'NAKDI_TOPLAM': 'Nakdi Toplam',
	'GNAKDI_TL_BAK': 'G.Nakdi TL Bakiye',
	'GNAKDI_YP_BAK': 'G.Nakdi YP Bakiye',
	'GNAKDI_TOPLAM': 'G.Nakdi Toplam',
	'NAKDI_TL_REESKONT': 'Nakdi TL Reeskontlu',
	'NAKDI_YP_REESKONT': 'Nakdi YP Reeskontlu',
	'NAKDI_REESKONTLU_TOPLAM': 'Nakdi Reeskontlu Toplam',
    'GUNCEL_NAKDI_KARSILIK': 'Nakdi Karşılık',
    'GUNCEL_GAYRI_NAKDI_KARSILIK': 'G.Nakdi Karşılık',
    'GUNCEL_TOPLAM': 'Toplam Karşılık',
}

kr202_sube_rename_dict = {
    'KKTC Şubeleri': 'Kıbrıs',
    'Karma Şubeler': 'Karma',
    'Kurumsal Şubeler': 'Kurumsal',
    'Ticari İhtisas Şubeleri': 'Ticari',
    'Y.dışı': 'Yurt Dışı',
    'Serbest Bölge Şubeleri': 'Serbest Bölge',
    'Özel Bankacılık Şubeleri': 'Özel Bankacılık',
}

kr202_bolge_rename_dict = {
    'K.K.T.C. MÜDÜRLÜĞÜ': 'KKTC Müdürlüğü',
    'KOBİ K.T.B.GAZİANTEP B.M': 'Gaziantep Bölge Müdürlüğü',
    'KOBİ K.T.B.İST-AVCIL.B.M': 'İstanbul-Avcılar Bölge Müdürlüğü',
    'KOBİ K.T.B.İST-MALTE.B.M': 'İstanbul-Maltepe Bölge Müdürlüğü',
    'KOBİ K.T.B.İST.-ŞİŞL.B.M': 'İstanbul-Şişli Bölge Müdürlüğü',
    'KOBİ K.T.B.İST.BAYRA.B.M': 'İstanbul-Bayrampaşa Bölge Müdürlüğü',
    'KOBİ K.TAH.B.ANTALYA B.M': 'Antalya Bölge Müdürlüğü',
    'KOBİ K.TAH.B.DENİZLİ B.M': 'Denizli Bölge Müdürlüğü',
    'KOBİ K.TAH.B.KOCAELİ.B M': 'Kocaeli Bölge Müdürlüğü',
    'KOBİ KR.TAH.B.ADANA.B.M': 'Adana Bölge Müdürlüğü',
    'KOBİ KRE.TAH.ANK.BÖL.MÜ.': 'Ankara Bölge Müdürlüğü',
    'KOBİ KRE.TAH.B.BURSA B.M': 'Bursa Bölge Müdürlüğü',
    'KOBİ KRE.TAH.B.TRABZ.B.M': 'Trabzon Bölge Müdürlüğü',
    'KOBİ KRE.TAH.İZM.BÖL.MÜ.': 'İzmir Bölge Müdürlüğü',
    'KOBİ KREDİ.B.TAH.B.DİYAR': 'Diyarbakır Bölge Müdürlüğü',
}

kr202_bolum_rename_dict = {
    'KKTB': 'Kurumsal Krediler Tahsis Bölümü',
    'Kurumsal Tahsis': 'Kurumsal Krediler Tahsis Bölümü',
    'Kurumsal Krediler Tahsis': 'Kurumsal Krediler Tahsis Bölümü',
    'Kurumsal Kredi Krediler Tahsis': 'Kurumsal Krediler Tahsis Bölümü',
    'Kurumsal Kredi Krediler Tahsis Bölümü': 'Kurumsal Krediler Tahsis Bölümü',
    'TKTB': 'Ticari Krediler Tahsis Bölümü',
    'Ticari Tahsis': 'Ticari Krediler Tahsis Bölümü',
    'Ticari Krediler Tahsis': 'Ticari Krediler Tahsis Bölümü',
    'Ticari Kredi Krediler Tahsis': 'Ticari Krediler Tahsis Bölümü',
    'Ticari Kredi Krediler Tahsis Bölümü': 'Ticari Krediler Tahsis Bölümü',
    'Perakende Tahsis': 'Perakende Krediler Tahsis Bölümü',
    'Krediler İzleme': 'Krediler İzleme Bölümü',
    'Takip': 'Ticari ve Kurumsal Krediler Takip Bölümü',
    'Proje Finansmanı': 'Proje Finansmanı Bölümü'
}

kr202_tsa_rename_dict = {
    'İNŞAAT': 'İNŞAAT VE TAAHHÜT',
    'İNŞAAT TAAHHÜT': 'İNŞAAT VE TAAHHÜT',
    'ULAŞTIRMA DEPOLAMA': 'ULAŞTIRMA VE DEPOLAMA',
    'OTOMOTİV ULAŞIM ARAÇLARI': 'OTOMOTİV VE ULAŞIM ARAÇLARI',
    'DİĞER HİZMET': 'DİĞER HİZMET FAALİYETLERİ',
    'ELEKTRONİK BİLİŞİM': 'ELEKTRONİK VE BİLİŞİM',
    'GEMİCİLİK TERSANECİLİK': 'GEMİCİLİK VE TERSANECİLİK',
    'SAVUNMA SANAYİ': 'SAVUNMA SANAYİİ',
    'PPP (Kamu Garantili KÖİ) (HASTANE VE OTOYOLLAR)': 'PPP (HASTANE VE OTOYOLLAR)',
    'RAFİNERİ VE YAKIT TİCARETİ (AKARYAKIT, PETROL, KÖMÜR, GAZ)': 'RAFİNERİ VE YAKIT TİCARETİ',
    'MAKİNE VE TEÇHİZAT SANAYİİ VE TİCARETİ (savaş araçları imalatı hariç)': 'MAKİNE VE TEÇHİZAT SANAYİİ VE TİCARETİ',
    'BELEDİYELER VE OSBler': 'BELEDİYELER VE OSBLER',
    'BELEDİYELER VE OSB ler': 'BELEDİYELER VE OSBLER',
    'BELEDİYELER VE OSBLER': 'BELEDİYELER VE OSBLER',
    'BELEDİYELER VE OSB LER': 'BELEDİYELER VE OSBLER',
    'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVMler)': 'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVMLER)',
    'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVM ler)': 'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVMLER)',
    'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVMLER)': 'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVMLER)',
    'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVM LER)': 'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVMLER)',
    'nan': np.nan,
    '-': np.nan,
}


def fix_kr202_df(df, i):
    df.columns = [x.strip() for x in df.columns]
    for x in kr202_column_rename_dict:
        try:
            df = df.rename(columns={x: kr202_column_rename_dict[x]})
        except:
            pass
    
    for x in kr202_float_list:
        if x in df.columns and df[x].dtype == 'O':
            df[x] = df[x].apply(lambda x: float(str(x).replace(',', '.')))

    df = df.loc[df['Müşteri No'].notnull()]
    if 'Toplam Risk' not in df.columns:
        df['Toplam Risk'] = df['Nakdi Toplam'] + df['G.Nakdi Toplam']
    if 'Toplam Reeskontlu Risk' not in df.columns:
        df['Toplam Reeskontlu Risk'] = df['Nakdi Reeskontlu Toplam'] + df['G.Nakdi Toplam']

    for c in ['Nakdi Karşılık', 'G.Nakdi Karşılık', 'Toplam Karşılık']:
        if c not in df.columns:
            df[c] = np.nan
        
    current_sube_tur_list = df['Şube Türü'].unique()
    if 'Kurumsal' in current_sube_tur_list and 'Kurumsal Şubeler' in current_sube_tur_list:
        df['Şube Türü'] = df['Şube Türü'].map(lambda x: {'Kurumsal': 'KKTB'}.get(x, x))
    df['Şube Türü'] = df['Şube Türü'].map(lambda x: kr202_sube_rename_dict.get(x, x))
    df['Şube Adı'] = df['Şube Adı'].apply(lambda x: x.strip().replace('I', 'ı').title().replace(' Ve ', ' ve ') if type(x) is str else np.nan)
    df['Bölge Adı'] = df['Bölge Adı'].apply(lambda x: x.strip() if type(x) is str else np.nan)
    df['Bölge Adı'] = df['Bölge Adı'].map(lambda x: kr202_bolge_rename_dict.get(x, x))

    if i == len(kr202_df_raw_list) - 1:
        df = df.loc[:, df.columns.isin(kr202_include_list)]
    else:
        df = df[kr202_initial_column_list]

    df['İlgili Bölüm'] = df['İlgili Bölüm'].map(lambda x: kr202_bolum_rename_dict.get(x, x))
    df['Tahsis Sektör Adı'] = df['Tahsis Sektör Adı'].fillna('-').apply(lambda x: x.strip().replace('\'', '').replace('  ', ' '))
    df['Tahsis Sektör Adı'] = df['Tahsis Sektör Adı'].map(lambda x: kr202_tsa_rename_dict.get(x, x))
    df['NACE Kodu'] = df['NACE Kodu'].apply(lambda x: x.strip() if type(x) is str else np.nan)
    df['NACE Harf'] = df['NACE Kodu'].apply(lambda x: x[0] if type(x) is str else np.nan)

    df['Nakdi Karşılık Oranı'] = df['Nakdi Karşılık'] / df['Nakdi Reeskontlu Toplam']
    df['G.Nakdi Karşılık Oranı'] = df['G.Nakdi Karşılık'] / df['G.Nakdi Toplam']
    df['Karşılık Oranı'] = df['Toplam Karşılık'] / df['Toplam Reeskontlu Risk']

    return df

    
df_kr202_all, df_kr202_last = batch_process_files(kr202_df_raw_list, kr202_sorted_list, kr202_sorted_date_list, fix_kr202_df, kr202_column_list)


customer_list = []
for df in df_kr202_all:
    customer_list.append(list(df['Müşteri No']))

df_kr202_last_original_backup = df_kr202_last.copy()

KR202 - 2023.12.31.xlsx işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## Müşteri Sınıfı

In [145]:
ms_file_list, ms_date_list, ms_sorted_list = scan_folder(ms_folder_path, '.csv')
ms_sorted_date_list, ms_df_raw_list = batch_import_files(ms_sorted_list, ms_folder_path, 'Müşteri Sınıfı', 'ms', csv=[';'])

MS - 2023.12.31.csv okunuyor
MS - 2023.12.31.csv okundu.
Tüm dosyalar başarıyla okundu


In [146]:
ms_column_list = ['MUSTERI_NO', 'MUSTERI_SINIFI']

renamed_ms_column_list = ['Müşteri No', 'Müşteri Sınıfı']

def fix_ms_df(df, i):
    df = df.loc[df['MUSTERI_NO'].notnull(), ms_column_list]
    df.columns = renamed_ms_column_list
    df['Müşteri Sınıfı'] = df['Müşteri Sınıfı'].apply(lambda x: int(x[-1]))
    return df


df_ms_all, df_ms_last = batch_process_files(ms_df_raw_list, ms_sorted_list, ms_sorted_date_list, fix_ms_df)

MS - 2023.12.31.csv işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## Proje

In [147]:
proje_file_list, proje_date_list, proje_sorted_list = scan_folder(proje_folder_path, '.csv')
proje_sorted_date_list, proje_df_raw_list = batch_import_files(proje_sorted_list, proje_folder_path, 'Proje Kredileri', 'proje', csv=[';', 'ANSI', False, False])

Proje - 2023.12.31.csv okunuyor
Proje - 2023.12.31.csv okundu.
Tüm dosyalar başarıyla okundu


In [148]:
proje_column_list = [' MUS_NO ']

renamed_proje_column_list = ['Müşteri No']

def fix_proje_df(df, i):
    df = df.loc[df[' MUS_NO '].notnull(), proje_column_list]
    df.columns = renamed_proje_column_list
    df = df.drop_duplicates(keep='first')
    df['Proje Kredisi'] = 1
    return df


df_proje_all, df_proje_last = batch_process_files(proje_df_raw_list, proje_sorted_list, proje_sorted_date_list, fix_proje_df)

Proje - 2023.12.31.csv işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## Memzuç

In [149]:
memzuc_file_list, memzuc_date_list, memzuc_sorted_list = scan_folder(memzuc_folder_path, '.csv', -1)
memzuc_sorted_date_list, memzuc_df_raw_list = batch_import_files(memzuc_sorted_list, memzuc_folder_path, 'Memzuç', 'memzuc', csv=[';'])

Memzuç - 2023.11.01.csv okunuyor
Memzuç - 2023.11.01.csv okundu.
Tüm dosyalar başarıyla okundu


In [150]:
memzuc_column_list = [
    'MUS_NO', 'BANKA_SAY',
    'SEKTOR_LIMIT', 'SEKTOR_RISK', 'BANKAMIZ_LIMIT', 'BANKAMIZ_RISK',
    'SEKTOR_LIMIT_NAKDI', 'SEKTOR_RISK_NAKDI', 'BANKAMIZ_LIMIT_NAKDI', 'BANKAMIZ_RISK_NAKDI',
    'SEKTOR_LIMIT_GNAKDI', 'SEKTOR_RISK_GNAKDI', 'BANKAMIZ_LIMIT_GNAKDI', 'BANKAMIZ_RISK_GNAKDI'
]

renamed_memzuc_column_list = [
    'Müşteri No', 'Banka Sayısı',
    'Memzuç Limit', 'Memzuç Risk', 'Bankamız Limit', 'Bankamız Risk',
    'Memzuç Nakdi Limit', 'Memzuç Nakdi Risk', 'Bankamız Nakdi Limit', 'Bankamız Nakdi Risk',
    'Memzuç G.Nakdi Limit', 'Memzuç G.Nakdi Risk', 'Bankamız G.Nakdi Limit', 'Bankamız G.Nakdi Risk'
]

def fix_memzuc_df(df, i):
    df = df.loc[df['MUS_NO'].notnull(), memzuc_column_list]
    df.columns = renamed_memzuc_column_list
    df = df.drop_duplicates(subset='Müşteri No', keep='first')
    df['Sektör Limit Doluluk Oranı'] = df['Memzuç Risk'] / df['Memzuç Limit']
    df['Bankamız Limit Doluluk Oranı'] = df['Bankamız Risk'] / df['Bankamız Limit']
    df['Bankamız Memzuç Risk Payı'] = df['Bankamız Risk'] / df['Memzuç Risk']
    df['Sektör Nakdi Limit Doluluk Oranı'] = df['Memzuç Nakdi Risk'] / df['Memzuç Nakdi Limit']
    df['Bankamız Nakdi Limit Doluluk Oranı'] = df['Bankamız Nakdi Risk'] / df['Bankamız Nakdi Limit']
    df['Bankamız Memzuç Nakdi Risk Payı'] = df['Bankamız Nakdi Risk'] / df['Memzuç Nakdi Risk']
    df.loc[:, [c for c in df.columns if c not in renamed_memzuc_column_list]] = df.loc[:, [c for c in df.columns if c not in renamed_memzuc_column_list]].clip(0, 100)   
    return df

df_memzuc_all, df_memzuc_last = batch_process_files(memzuc_df_raw_list, memzuc_sorted_list, memzuc_sorted_date_list, fix_memzuc_df)

Memzuç - 2023.11.01.csv işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## Yapay Zeka

In [151]:
yz_file_list, yz_date_list, yz_sorted_list = scan_folder(yz_folder_path, '.csv')
yz_sorted_date_list, yz_df_raw_list = batch_import_files(yz_sorted_list, yz_folder_path, 'İzleme', 'yz', csv=[';'])

İzleme - 2023.12.31.csv okunuyor
İzleme - 2023.12.31.csv okundu.
Tüm dosyalar başarıyla okundu


In [152]:
yz_column_list = ['MUSTERI_NUMARASI', 'YZ_RISK_SINIFI', 'KM_RISK_SINIFI', 'IZLEME_RISK_SINIFI', 'MODEL', 'GECIKME_GUN_SAYISI']

yz_column_rename_dict = {
    'MUSTERI_NUMARASI': 'Müşteri No',
    'YZ_RISK_SINIFI': 'Yapay Zeka Risk Sınıfı',
    'KM_RISK_SINIFI': 'Karar Motoru Risk Sınıfı',
    'IZLEME_RISK_SINIFI': 'İzleme Risk Sınıfı',
    'MODEL': 'Gecikme',
    'GECIKME_GUN_SAYISI': 'Gecikme Gün'
}

def fix_yz_df(df, i):
    df = df.loc[df['MUSTERI_NUMARASI'].isin(customer_list[i]), yz_column_list]
    df = df.rename(columns = yz_column_rename_dict)
    # df['Gecikme'] = df['Gecikme'].map(yz_gecikme_map_dict)
    df['Gecikme'] = df['Gecikme'].apply(lambda x: {'Tahsilat': 1}.get(x, 0))
    return df


df_yz_all, df_yz_last = batch_process_files(yz_df_raw_list, yz_sorted_list, yz_sorted_date_list, fix_yz_df)

İzleme - 2023.12.31.csv işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## Entegre Derece Skor

In [153]:
eds_file_list, eds_date_list, eds_sorted_list = scan_folder(eds_folder_path, '.csv')
eds_sorted_date_list, eds_df_raw_list = batch_import_files(eds_sorted_list, eds_folder_path, 'Entegre Derece Skor', 'eds', csv=[';'])

EDS - 2023.12.01.csv okunuyor
EDS - 2023.12.01.csv okundu.
Tüm dosyalar başarıyla okundu


In [154]:
eds_column_list = ['MUSTERI_NO', 'CUSTOMER_TYPE', 'FINAL_RATING']
renamed_eds_column_list = ['Müşteri No', 'Değerlendirmeye Esas Segment', 'Entegre Derece Skor']

des_map_dict = {
    'TARIM': 'İŞLETME',
    'ISLETME': 'İŞLETME',
    'KOBI': 'KOBİ',
    'TİCARİ': 'TİCARİ',
    'KURUMSAL': 'KURUMSAL',
}

def fix_eds_df(df, i):
    df = df.loc[df['MUSTERI_NO'].notnull(), eds_column_list]
    df.columns = renamed_eds_column_list
    df['Değerlendirmeye Esas Segment'] = df['Değerlendirmeye Esas Segment'].map(des_map_dict)
    return df

df_eds_all, df_eds_last = batch_process_files(eds_df_raw_list, eds_sorted_list, eds_sorted_date_list, fix_eds_df)

EDS - 2023.12.01.csv işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## Finansal Güçlük

In [155]:
fg_file_list, fg_date_list, fg_sorted_list = scan_folder(fg_folder_path)
fg_sorted_date_list, fg_df_raw_list = batch_import_files(fg_sorted_list, fg_folder_path, 'Finansal Güçlük', 'fg', True)

Finansal Güçlük - 2023.12.01.xlsx önbellekten okundu (100%)
Tüm dosyalar başarıyla okundu


In [156]:
fg_column_list = [
    'Müşteri\nNumarası',
    'Çek\nYasaklılık',
    'Son 2 Yıldaki\nMemzuç Dönemlerinden \nHerhangi Birinde Diğer \nBanka Takip > 250 TL\n',
    'Ticari Sicil\nPaylaşım Sistemi\nİflas/Konkordato/\nİflas Erteleme',
    'Son 1 Yıldaki Memzuç\nDönemlerinden Herhangi \nBirinde Diğer Banka \nTazmin > 1000 TL',
    'Diğer Banka \nCanlı Alacaklardan \nYeniden Yapılandırma > 0',
    'Diğer Banka Canlı Alacaklardan \nYeniden Yapılandırma/\nDiğer Banka Toplam \nNakdi Risk Oranı (%)\n',
    'Bankamız\nKarşılıksız\nÇek',
    'Diğer Banka\nKarşılıksız\nÇek',
    'Bankamız ve\nDiğer Banka\nKarşılıksız Çek',
    'Diğer Banka\nTahakkuk\n>20.000',
    'Diğer Toplam Tahakkuk /\nDiğer Toplam Nakdi Risk\n> 0,01'
]

fg_new_column_list = [
    'Müşteri\nNumarası',
    'Çek\nYasaklılık',
    'Karşılıksız Çek',
    'Diğer Banka Tahakkuk',
    'Son 2 Yıldaki\nMemzuç Dönemlerinden \nHerhangi Birinde Diğer \nBanka Takip > 250 TL\n',
    'Ticari Sicil\nPaylaşım Sistemi\nİflas/Konkordato/\nİflas Erteleme',
    'Son 1 Yıldaki Memzuç\nDönemlerinden Herhangi \nBirinde Diğer Banka \nTazmin > 1000 TL',
    'Diğer Banka Yapılandırma',
    'Diğer Banka Canlı Alacaklardan \nYeniden Yapılandırma/\nDiğer Banka Toplam \nNakdi Risk Oranı (%)\n'
]

renamed_fg_column_list = [
    'Müşteri No',
    'Çek Yasaklılık',
    'Karşılıksız Çek',
    'Diğer Banka Tahakkuk',
    'Diğer Banka Takip',
    'TSPS İflas vb',
    'Diğer Banka Tazmin',
    'Diğer Banka Yapılandırma',
    'Diğer Banka Yapılandırma Oranı (%)'
]


def check_karsiliksiz_cek(df):
    if df['Bankamız\nKarşılıksız\nÇek'] == 1 or df['Diğer Banka\nKarşılıksız\nÇek'] == 1 or df['Bankamız ve\nDiğer Banka\nKarşılıksız Çek'] == 1:
        return 1
    return 0

def check_tahakkuk(df):
    if df['Diğer Banka\nTahakkuk\n>20.000'] == 1 or df['Diğer Toplam Tahakkuk /\nDiğer Toplam Nakdi Risk\n> 0,01'] == 1:
        return 1
    return 0
    
def check_yapilandirma(df):
    if df['Diğer Banka \nCanlı Alacaklardan \nYeniden Yapılandırma > 0'] == 1:
        return 1
    return 0


def fix_fg_df(df, i):
    df = df.loc[df['Müşteri\nNumarası'].notnull(), fg_column_list]

    df = df.replace('VAR', 1).replace(['YOK', np.nan], 0)

    df['Karşılıksız Çek'] = df.apply(check_karsiliksiz_cek, axis = 1)
    df['Diğer Banka Tahakkuk'] = df.apply(check_tahakkuk, axis = 1)
    df['Diğer Banka Yapılandırma'] = df.apply(check_yapilandirma, axis = 1)

    df = df[fg_new_column_list]
    df.columns = renamed_fg_column_list

    df['FG Kriter Sayısı'] = 0
    for col in df.columns[1:-2]:
        df[col] = df[col].fillna(0)
        df['FG Kriter Sayısı'] += df[col]

    df['Finansal Güçlük'] = df['FG Kriter Sayısı'].apply(lambda x: 1 if x > 0 else 0)

    return df


df_fg_all, df_fg_last = batch_process_files(fg_df_raw_list, fg_sorted_list, fg_sorted_date_list, fix_fg_df)

Finansal Güçlük - 2023.12.01.xlsx işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## KRÖA

In [157]:
kroa_file_list, kroa_date_list, kroa_sorted_list = scan_folder(kroa_folder_path, '.csv')
kroa_sorted_date_list, kroa_df_raw_list = batch_import_files(kroa_sorted_list, kroa_folder_path, 'KRÖA', 'kroa', csv=[';'])

KRÖA - 2023.12.31.csv okunuyor
KRÖA - 2023.12.31.csv okundu.
Tüm dosyalar başarıyla okundu


In [158]:
kroa_column_list = ['Müşteri No', 'KRÖA']

def fix_kroa_df(df, i):
    dfn = pd.DataFrame()
    dfn['Müşteri No'] = df[' MUSTERI_NO'].unique()
    dfn = dfn.loc[dfn['Müşteri No'].notnull()]
    dfn['KRÖA'] = 1
    return dfn

df_kroa_all, df_kroa_last = batch_process_files(kroa_df_raw_list, kroa_sorted_list, kroa_sorted_date_list, fix_kroa_df, kroa_column_list)

KRÖA - 2023.12.31.csv işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


## LGD

In [159]:
lgd_file_list, lgd_date_list, lgd_sorted_list = scan_folder(lgd_folder_path, '.csv')
lgd_sorted_date_list, lgd_df_raw_list = batch_import_files(lgd_sorted_list, lgd_folder_path, 'LGD', 'lgd', csv=[';'])

LGD - 2023.12.01.csv okunuyor
LGD - 2023.12.01.csv okundu.
Tüm dosyalar başarıyla okundu


In [160]:
lgd_column_list = ['CUST_ID', 'LGD_IFRS']

renamed_lgd_column_list = ['Müşteri No', 'LGD']

def fix_lgd_df(df, i):
    df = df.loc[df['CUST_ID'].notnull(), lgd_column_list]
    df.columns = renamed_lgd_column_list
    df['LGD'] = df['LGD'].apply(lambda x: float(x.replace(',', '.')) if type(x) is str else x)
    return df

df_lgd_all, df_lgd_last = batch_process_files(lgd_df_raw_list, lgd_sorted_list, lgd_sorted_date_list, fix_lgd_df)

LGD - 2023.12.01.csv işlendi (100%)
Tüm dosyalar başarıyla oluşturuldu


# Veri Birleştirme

## df_wide

In [161]:
sorted_date_list = yz_sorted_date_list

df = df_kr202_last_original_backup.copy()
df = pd.merge(df, df_ms_last, on='Müşteri No', how='left')
df = pd.merge(df, df_proje_last, on='Müşteri No', how='left')
df['Proje Kredisi'] = df['Proje Kredisi'].fillna(0)
# df = df.loc[df['Müşteri Sınıfı'] <= 2]
df = df.loc[df['İlgili Bölüm'].notnull()] 
df_kr202_last = df.copy()

df = df_eds_last.copy()
df = pd.merge(df, df_yz_last, how='outer', on='Müşteri No')
df = pd.merge(df, df_kroa_last, how='outer', on='Müşteri No')
df = pd.merge(df, df_lgd_last, how='outer', on='Müşteri No')
df_izleme_last = df.copy()

## df_master

In [162]:
master_column_list = [
    'Müşteri No', 'Adı', 'Şube Kodu', 'Şube Adı', 'Şube İl', 'Şube İlçe',
    'Şube Türü', 'Bölge Adı', 'Yetki Kodu',
    'İlgili Bölüm', 'Müşteri Sınıfı', 'Risk Grubu Kodu', 'Risk Grubu Adı',
    'Tahsis Sektör Adı', 'Tahsis Alt Sektör Adı',
    'NACE Kodu', 'NACE Harf', 'Sektör Derece İzleme',
    'Nakdi TL Bakiye', 'Nakdi YP Bakiye', 'Nakdi Toplam',
    'G.Nakdi TL Bakiye', 'G.Nakdi YP Bakiye', 'G.Nakdi Toplam', 'Toplam Risk',
    'Nakdi TL Reeskontlu', 'Nakdi YP Reeskontlu', 'Nakdi Reeskontlu Toplam', 'Toplam Reeskontlu Risk',
    'Nakdi Karşılık', 'G.Nakdi Karşılık', 'Toplam Karşılık',
    'Nakdi Karşılık Oranı', 'G.Nakdi Karşılık Oranı', 'Karşılık Oranı',
    'Proje Kredisi', 'Sektör Derecesi'
]

dfm = df_kr202_last.copy()

dfm['NACE 6lı'] = dfm['NACE Kodu'].apply(lambda x: x[2:] if type(x) is str else np.nan)
dfm = pd.merge(dfm, df_sektor, on='NACE 6lı', how='left')

for df in [df_kur, df_kroa_yaklasma, df_izleme_last, df_fg_last, df_memzuc_last, df_mali]:
    dfm = pd.merge(dfm, df, on='Müşteri No', how='left')
    cl = list(df.columns)
    if 'Müşteri No' in cl:
        cl.remove('Müşteri No')
    master_column_list += cl

for c in list(df_fg_last.columns) + ['KRÖA', 'KRÖA Yaklaşma']:
    dfm[c] = dfm[c].fillna(0)

dfm = dfm.sort_values('Nakdi Reeskontlu Toplam', ascending=False).reset_index(drop=True)

dfm = dfm[master_column_list]

df_master = dfm.copy()

# Listeler

In [163]:
df = df_kr202_last.copy()

segment_list = ['KURUMSAL', 'TİCARİ', 'KOBİ', 'İŞLETME']
segment_title_list = ['Kurumsal', 'Ticari', 'KOBİ', 'İşletme']
segment_title_dict = dict(zip(segment_list, segment_title_list))

bolum_list = [
    'Kurumsal Krediler Tahsis Bölümü',
    'Ticari Krediler Tahsis Bölümü',
    'Perakende Krediler Tahsis Bölümü',
    'Proje Finansmanı Bölümü',
    'Krediler İzleme Bölümü',
    'Ticari ve Kurumsal Krediler Takip Bölümü',
]
filtered_bolum_list = [
    'Kurumsal Krediler Tahsis Bölümü',
    'Ticari Krediler Tahsis Bölümü',
    'Perakende Krediler Tahsis Bölümü',
]

sektor_list = [x for x in df['Tahsis Sektör Adı'].unique() if x not in ['', ' ', np.nan, 0]]
sektor_list.sort(key=locale.strxfrm)
filtered_sektor_list = [x for x in sektor_list if x not in ['PPP (Kamu Garantili KÖİ) (HASTANE VE OTOYOLLAR)', 'FİNANS']]

sektor_title_dict = {}
for s in sektor_list:
    sektor_title_dict[s] = s.replace('I', 'ı').title().replace('Ve', 've').replace('Ppp', 'PPP').replace('Avmler', 'AVMler').replace('Osbler', 'OSBler')
sektor_title_short_dict = sektor_title_dict.copy()
sektor_title_short_dict = {
    **sektor_title_short_dict, **{
        'BELEDİYELER VE OSBLER': 'Beledi̇ye ve OSB',
        'DERİ VE DERİ ÜRÜNLERİ': 'Deri ve Deri Ürün.',
        'DİĞER HİZMET FAALİYETLERİ': 'Diğer Hizmet Faal.',
        'GEMİCİLİK VE TERSANECİLİK': 'Gemicilik ve Tersa.',
        'MAKİNE VE TEÇHİZAT SANAYİİ VE TİCARETİ': 'Mak. ve Teçhi. San.',
        'OTOMOTİV VE ULAŞIM ARAÇLARI': 'Oto. ve Ulaş. Araç.',
        'PPP (HASTANE VE OTOYOLLAR)': 'PPP (Hast. ve Oto.)',
        'RAFİNERİ VE YAKIT TİCARETİ': 'Rafineri ve Yakıt Tic.',
        'TARIM VE HAYVANCILIK': 'Tarım ve Hayvanc.',
        'TAŞA TOPRAĞA DAYALI SANAYİ': 'Taşa Topr. Day. San.',
        'TİCARİ GAYRİMENKUL İŞLETMECİLİĞİ (AVMLER)': 'Ticari Gayrim. İşl.',
        'TÜTÜN VE TÜTÜN ÜRÜNLERİ': 'Tütün ve Tütün Ür.',
        'ULAŞTIRMA VE DEPOLAMA': 'Ulaştırma ve Depo.',
    }
}

sube_list = [x for x in sorted(df['Şube Kodu'].unique()) if x >= 1000] 
dft = df[['Şube Kodu', 'Şube Adı']].drop_duplicates('Şube Kodu', keep='first').reset_index(drop='True')
sube_kodu_adi_dict = dict(zip(dft['Şube Kodu'], dft['Şube Adı']))
sube_kodu_adi_dict = {**sube_kodu_adi_dict, **{'TOPLAM': 'TOPLAM'}}
sube_adi_kodu_dict = {v: k for k, v in sube_kodu_adi_dict.items()}

bolge_list = [x for x in df['Bölge Adı'].unique() if x not in ['', ' ', np.nan, 0]] 
bolge_list.sort(key=locale.strxfrm)

list_list = (segment_list, segment_title_list, segment_title_dict, bolum_list, filtered_bolum_list, sektor_list, filtered_sektor_list, sektor_title_dict, sektor_title_short_dict, sube_list, bolge_list)
save_pickle(list_list, 'list_list')

# Portföy Risklilik Analizi

## Kriter Fonksiyonları

In [164]:
def fg_score(df):
    return int(df['FG Kriter Sayısı'])

def eds_score(df):
    if df['Entegre Derece Skor'] == 12: return 10
    elif df['Entegre Derece Skor'] == 11: return 8
    elif df['Entegre Derece Skor'] == 10: return 6
    elif df['Entegre Derece Skor'] == 9: return 4
    elif df['Entegre Derece Skor'] == 8: return 3
    elif df['Entegre Derece Skor'] == 7: return 2
    elif df['Entegre Derece Skor'] == 6: return 1
    return 0

def yz_score(df):
    if df['Yapay Zeka Risk Sınıfı'] == 5: return 10 + df['Gecikme'] * 6
    elif df['Yapay Zeka Risk Sınıfı'] == 4: return 8 + df['Gecikme'] * 4
    elif df['Yapay Zeka Risk Sınıfı'] == 3: return 6 + df['Gecikme'] * 2
    elif df['Yapay Zeka Risk Sınıfı'] == 2: return 4 + df['Gecikme'] * 1
    return 0

def kur_riski_score(df):
    if df['Kur Riski Çalışması Sınıfı'] == 'Yüksek Riskli': return 4
    elif df['Kur Riski Çalışması Sınıfı'] == 'Yakından İzlenmeli': return 2
    return 0

def kvfb_score(df):
    if df['KVFB / Net Satışlar'] > 2: return 2
    elif df['KVFB / Net Satışlar'] > 1: return 1
    return 0

def borclanma_score(df):
    if df['Borçlanma Oranı'] > 8 or df['YENI_OZSERMAYE'] < 0: return 2
    elif df['Borçlanma Oranı'] > 4: return 1
    return 0

def kaldirac_score(df):
    riskli_sektor_list = [
        'İNŞAAT VE TAAHHÜT',
        'BELEDİYELER VE OSBLER',
        'MADENCİLİK',
        'GEMİCİLİK VE TERSANECİLİK',
        'TURİZM',
    ]
    if df['Tahsis Sektör Adı'] in riskli_sektor_list:
        low, high = 4, 12
    else:
        low, high = 8, 14
    if df['Kaldıraç Oranı'] > high or df['EBITDA'] < 0: return 2
    elif df['Kaldıraç Oranı'] > low: return 1
    return 0

def faiz_yuku_score(df):
    if df['Faiz Yükü Karşılama Farkı'] < -6: return 2
    elif df['Faiz Yükü Karşılama Farkı'] < 0: return 1
    return 0

def cari_oran_score(df):
    if df['Cari Oran'] < 1: return 2
    elif df['Cari Oran'] < 2: return 1
    return 0

def sektor_score(df):
    if df['Sektör Derecesi'] in ['C+', 'C', 'C-', 'D+', 'D', 'D-']: return 6
    elif df['Sektör Derecesi'] == 'B-': return 3
    return 0

def gecikme_score(df):
    if df['Gecikme Gün'] >= 21: return 7
    elif df['Gecikme Gün'] >= 11: return 5
    elif df['Gecikme Gün'] >= 1: return 3
    return 0

def mdr_score(df):
    if df['TSPS İflas vb'] or df['Diğer Banka Takip'] or df['Diğer Banka Tahakkuk'] or df['Diğer Banka Tazmin']:
        return np.inf
    elif df['Gecikme Gün'] >= 10: return np.inf
    return 0

def kroa_score(df):
    if df['KRÖA Yaklaşma']: return 4
    return 0

## Risk Kategori Fonksiyonları

In [165]:
def categorize_pra(df, x=None, threshold_list=None, category_list=None):
    default_x = 'Toplam Puan'
    default_threshold_list = [0, 2, 5, 10, 15, np.inf]
    default_category_list = ['Mükemmel', 'Çok Düşük', 'Düşük', 'Orta', 'Yüksek', 'Çok Yüksek']

    if x is None:
        x = default_x
    if threshold_list is None:
        threshold_list = default_threshold_list
    if category_list is None:
        category_list = default_category_list

    if df[x] == np.inf: return category_list[-1]
    for t, c in zip(threshold_list, category_list):
        if df[x] <= t: return c
    return category_list[-1]
    
def create_pra_risk_categories(dfp, x='Toplam Puan', threshold_list=None, category_list=None, r='Risk Kategorisi', xx=None, factor=1_000_000):
    m = 'Müşteri No'
    c = 'Nakdi Reeskontlu Toplam'
    nn = 'Adet'
    s = 'Kümülatif Risk'
    ss = 'Kümülatif Adet'
    p = '%'
    pp = '% '
    if xx is None: xx=x

    df = dfp[[x, c]].copy()
    df = df.groupby(x).sum().reset_index()
    df[c] /= factor
    df[s] = df[c].cumsum()
    t = df[s].values[-1]
    df[p] = df[s] / t
    df[r] = df.apply(categorize_pra, x=xx, threshold_list=threshold_list, category_list=category_list, axis=1)
    
    dft = dfp[[m, x]].copy()
    dft = dft.groupby(x).count().reset_index().rename(columns={m: nn})
    dft[ss] = dft[nn].cumsum()
    tt = dft[ss].values[-1]
    dft[pp] = dft[ss] / tt

    df[' '] = np.nan
    df = pd.merge(df, dft, on=x)

    return df


def return_excluded_info(name, df, minus=True, factor=1_000_000):
    sign = -1 if minus else 1
    return [name, sign * df[mn].count(), sign * df[nr].sum() / factor, sign * df[gnr].sum() / factor, sign * df[tr].sum() / factor]

def return_percentages(c):
    percentages = list(dfk.loc[dfk['Küme'] == c].values[0][1:] / dfk.loc[dfk['Küme'] == c0].values[0][1:])
    return [' '] + percentages

## Analiz Fonksiyonları

In [166]:
puan_column_list = [
    'Finansal Güçlük Puanı', 'Entegre Derece Puanı',
    'Risk Sınıfı Puanı', 'Kur Riski Puanı',
    'KVFB / Net Satışlar Puanı', 'Borçlanma Oranı Puanı',
    'Kaldıraç Oranı Puanı', 'Faiz Yükü Karşılama Puanı', 'Cari Oran Puanı',
    'Sektör Riskliliği Puanı', 'Gecikme Puanı'
    'Öncül Risklilik Puanı (D.B. Tahakkuk, Takip, Tazmin, İflas, 10+ Gecikme)'
]

puan_fun_list = [
    fg_score, eds_score,
    yz_score, kur_riski_score,
    kvfb_score, borclanma_score,
    kaldirac_score, faiz_yuku_score, cari_oran_score,
    sektor_score, gecikme_score,
    mdr_score
]


def create_pra_analysis(df_pra):
    df = df_pra.copy()
        
    for c, f in zip(puan_column_list, puan_fun_list):
        df[c] = df.apply(f, axis=1)

    df['Toplam Puan'] = 0
    for c in puan_column_list:
        df['Toplam Puan'] += df[c]

    dfo = create_pra_risk_categories(df, 'Toplam Puan')
    df = pd.merge(df, dfo[['Toplam Puan', 'Risk Kategorisi']], on='Toplam Puan', how='left')

    df['Takıldığı Toplam Kriter Sayısı'] = 0
    for c in puan_column_list:
        df['Takıldığı Toplam Kriter Sayısı'] += df[c].apply(lambda x: min(1, x))

    dfyr = df.copy()
    dfyr = dfyr.loc[dfyr['Risk Kategorisi'].isin(['Yüksek', 'Çok Yüksek'])]

    df = pd.merge(df, dfyr.loc[:, ['Müşteri No'] + [x for x in list(dfyr.columns) if x not in list(df.columns)]], on='Müşteri No', how='left')
    dfyr = dfyr.loc[dfyr['Risk Kategorisi'].isin(['Çok Yüksek', 'Yüksek'])]

    return df.copy(), dfyr.copy(), dfo.copy()

## Sunum Fonksiyonları

### Özet

In [167]:
def create_pra_summary(df_pra):
    dfp = df_pra.copy()
    rk_list = ['Mükemmel', 'Çok Düşük', 'Düşük', 'Orta', 'Yüksek', 'Çok Yüksek']
    rk = 'Risk Kategorisi'
    factor = 1_000_000

    t_list = ['Genel Toplam']
    df = pd.DataFrame()
    df[rk] = rk_list
    df = add_v_space(df)
    t_list.append(np.nan)

    c = 'Müşteri No'
    n = 'Adet'
    p = 'Adet Payı'
    dft = dfp[[rk, c]].groupby(rk).count().reset_index().rename(columns={c: n})
    df = pd.merge(df, dft, on=rk, how='left')
    t = df[n].sum()
    t_list.append(t)
    df[p] = df[n] / t
    t_list.append(1)

    df = add_v_space(df)
    t_list.append(np.nan)

    c = 'Nakdi Reeskontlu Toplam'
    n = 'Nakdi Risk'
    p = 'Nakdi Risk Payı'
    dft = dfp[[rk, c]].groupby(rk).sum().reset_index().rename(columns={c: n})
    dft[n] /= factor
    df = pd.merge(df, dft, on=rk, how='left')
    t = df[n].sum()
    t_list.append(t)
    df[p] = df[n] / t
    t_list.append(1)

    df = add_v_space(df)
    t_list.append(np.nan)

    c = 'Entegre Derece Skor'
    n = 'Ortalama Entegre Derece / Skor'
    dft = dfp[[rk, c]].groupby(rk).mean().reset_index().rename(columns={c: n})
    df = pd.merge(df, dft, on=rk, how='left')
    t = dfp[c].mean()
    t_list.append(t)

    c = 'Yapay Zeka Risk Sınıfı'
    n = 'Ortalama Yapay Zeka Risk Sınıfı'
    dft = dfp[[rk, c]].groupby(rk).mean().reset_index().rename(columns={c: n})
    df = pd.merge(df, dft, on=rk, how='left')
    t = dfp[c].mean()
    t_list.append(t)

    df = add_h_space(df)
    df.loc[len(df)] = t_list
    df.loc[len(df)] = [f'{last_date} KR202, Reeskontlu, mio TL'] + [np.nan] * (len(df.columns) - 1)

    return insert_corner(df)

### Dağılım

In [168]:
def create_pra_dist(df_pra, rows, row_list, use_thresholds=False, reorder=False, rename_dict=None, factor=1_000_000):
    n_list = ['Mükemmel', 'Çok Düşük', 'Düşük', 'Orta', 'Yüksek', 'Çok Yüksek']
    rk = 'Risk Kategorisi'
    yrc = 'Yüksek Riskli Portföy Oranı'
    mn = 'Müşteri No'
    nrt = 'Nakdi Reeskontlu Toplam'
    factor = 1_000_000
    c = rows
    c_list = row_list
    dfp = df_pra.copy()
    dfs_list = []

    for x, find_risk, cc0 in zip([mn, nrt], [False, True], ['Adede Göre', 'Nakdi Riske Göre']):
        df = pd.DataFrame()
        if use_thresholds:
            ettl = c_list
            etl = [f'{et[0]} - {et[1]}' for et in ettl]
            df[c] = etl
        else:
            df[c] = c_list if c_list is not None else sorted(df_pra.loc[df_pra[c].notnull(), c].unique())
            
        df = add_v_space(df)
        t_list = ['Genel Toplam', np.nan]

        for n in n_list:
            dft = dfp.copy()
            if find_risk:
                dft = dft.loc[dft[rk] == n, [x, c]].groupby(c).sum().reset_index().rename(columns={x: n})
            else:
                dft = dft.loc[dft[rk] == n, [x, c]].groupby(c).count().reset_index().rename(columns={x: n})
            if use_thresholds:
                nx_list = []
                for et in ettl:
                    nx = dft.loc[(dft[c] >= et[0]) & (dft[c] <= et[1]), n].sum()
                    if nx == 0:
                        nx = np.nan
                    nx_list.append(nx)
                df[n] = nx_list
            else:
                df = pd.merge(df, dft, on=c, how='outer')
            t_list.append(df[n].sum())

        n = 'Toplam'
        dft = dfp.copy()
        if find_risk:
            dft = dft[[x, c]].groupby(c).sum().reset_index().rename(columns={x: n})
        else:
            dft = dft[[x, c]].groupby(c).count().reset_index().rename(columns={x: n})
        if use_thresholds:
            nx_list = []
            for et in ettl:
                nx = dft.loc[(dft[c] >= et[0]) & (dft[c] <= et[1]), n].sum()
                if nx == 0:
                    nx = np.nan
                nx_list.append(nx)
            df[n] = nx_list
        else:
            df = pd.merge(df, dft, on=c, how='outer')
        t_list.append(df[n].sum())

        df = df.dropna(axis=0, how='all', subset=list(df.columns)[1:], ignore_index=True)
        if rename_dict is not None:
            df[c] = df[c].map(rename_dict)
        df = add_h_space(df)
        df.loc[len(df)] = t_list
        if find_risk:
            df.iloc[:,1:] = df.iloc[:,1:] / factor
        df = add_v_space(df)
        df[yrc] = df[['Yüksek', 'Çok Yüksek']].sum(axis=1) / df['Toplam']
        if reorder:
            df.iloc[:-1, :] = df.iloc[:-1, :].sort_values(yrc, ascending=False)

        dfy = df.copy()
        for i in list(dfy.columns)[2:-2]:
            dfy[i] /= dfy[n]
        df = insert_header(df, cc0)
        dfy = insert_header(dfy, cc0)
        end = 'Reeskontlu, mio TL' if find_risk else 'Adet'
        df.loc[len(df)] = [f'{last_date} KR202, {end}'] + [np.nan] * (len(df.columns) - 1)
        dfs = h_stack([df, dfy])
        dfs_list.append(dfs)
        
    return insert_corner(v_stack(dfs_list))

### Cross Dağılım

In [169]:
def create_pra_dist_alt(df_pra, rows, row_list, use_thresholds=False, reorder=False, rename_dict=None, factor=1_000_000, find_risk=False):
    n_list = ['Mükemmel', 'Çok Düşük', 'Düşük', 'Orta', 'Yüksek', 'Çok Yüksek']
    rk = 'Risk Kategorisi'
    yrc = 'Yüksek Riskli Portföy Oranı'
    mn = 'Müşteri No'
    nrt = 'Nakdi Reeskontlu Toplam'
    factor = 1_000_000
    c = rows
    c_list = row_list
    dfp = df_pra.copy()
    x = nrt if find_risk else mn

    df = pd.DataFrame()
    if use_thresholds:
        ettl = c_list
        etl = [f'{et[0]} - {et[1]}' for et in ettl]
        df[c] = etl
    else:
        df[c] = c_list if c_list is not None else sorted(df_pra.loc[df_pra[c].notnull(), c].unique())
        
    t_list = ['Genel Toplam']

    for n in n_list:
        dft = dfp.copy()
        if find_risk:
            dft = dft.loc[dft[rk] == n, [x, c]].groupby(c).sum().reset_index().rename(columns={x: n})
        else:
            dft = dft.loc[dft[rk] == n, [x, c]].groupby(c).count().reset_index().rename(columns={x: n})
        if use_thresholds:
            nx_list = []
            for et in ettl:
                nx = dft.loc[(dft[c] >= et[0]) & (dft[c] <= et[1]), n].sum()
                nx_list.append(nx)
            df[n] = nx_list
        else:
            df = pd.merge(df, dft, on=c, how='outer')
        t_list.append(df[n].sum())

    n = 'Toplam'
    dft = dfp.copy()
    if find_risk:
        dft = dft[[x, c]].groupby(c).sum().reset_index().rename(columns={x: n})
    else:
        dft = dft[[x, c]].groupby(c).count().reset_index().rename(columns={x: n})
    if use_thresholds:
        nx_list = []
        for et in ettl:
            nx = dft.loc[(dft[c] >= et[0]) & (dft[c] <= et[1]), n].sum()
            nx_list.append(nx)
        df[n] = nx_list
    else:
        df = pd.merge(df, dft, on=c, how='outer')
    t_list.append(df[n].sum())

    df = df.dropna(axis=0, how='all', subset=list(df.columns)[1:], ignore_index=True)
    if rename_dict is not None:
        df[c] = df[c].map(rename_dict)
    df = add_h_space(df)
    df.loc[len(df)] = t_list
    if find_risk:
        df.iloc[:,1:] = df.iloc[:,1:] / factor
    df[yrc] = df[['Yüksek', 'Çok Yüksek']].sum(axis=1) / df['Toplam']
    if reorder:
        df.iloc[:-1, :] = df.iloc[:-1, :].sort_values(yrc, ascending=False)


    return df[[c, yrc]].copy()


def create_pra_cross_risk(df_pra, rows, columns, row_list, column_list, row_new=None, column_new=None, drop_na=True):
    r = rows
    r_list = row_list
    r_dict = row_new
    c = columns
    c_list = column_list
    n_list = column_new
    pg = 'Portföy Geneli'
    yrc = 'Yüksek Riskli Portföy Oranı'
    dfs = []

    for find_risk in [False, True]:
        df = pd.DataFrame()
        df[r] = r_list + [np.nan, 'Genel Toplam']
        df = add_v_space(df)

        for i, cc in enumerate(c_list):
            dfp = df_pra.copy()
            dfp = dfp.loc[dfp[c] == cc]
            dft = create_pra_dist_alt(dfp, r, r_list, False, False, find_risk=find_risk)
            n = n_list[i] if n_list is not None else cc
            dft.columns = [r, n ]
            df = pd.merge(df, dft, on=r, how='left')

        df = add_v_space(df)

        dfp = df_pra.copy()
        dft = create_pra_dist_alt(dfp, r, r_list, False, False)
        dft.columns = [r, pg]
        how = 'inner' if drop_na else 'left'
        df = pd.merge(df, dft, on=r, how=how)

        if r_dict is not None:
            df[r] = df[r].apply(lambda x: sektor_title_dict.get(x, x))
        
        hs = 'Nakdi Riske Göre' if find_risk else 'Adede Göre'
        df = insert_header(df, hs + ' ' + yrc)
        end = 'Reeskontlu, mio TL' if find_risk else 'Adet'
        df.loc[len(df)] = [f'{last_date} KR202, {end}'] + [np.nan] * (len(df.columns) - 1)
        dfs.append(df.copy())

    return insert_corner(h_stack(dfs))

### Adet Risk Dağılımı

In [170]:
def create_pra_count_risk(df_pra, rows, row_list, row_new, drop_na=True, sort='Adet', high_risk_only=True, factor=1_000_000):
    dfp = df_pra.copy()
    dfp = dfp.loc[dfp['Risk Kategorisi'].isin(['Çok Yüksek', 'Yüksek'])]
    r = rows
    r_list = row_list
    r_dict = row_new

    df = pd.DataFrame()
    df[r] = r_list
    df = add_v_space(df)
    t_list = ['Genel Toplam', np.nan]

    for c, n, p, fr in zip(['Müşteri No', 'Nakdi Reeskontlu Toplam'], ['Adet', 'Nakdi Risk'], ['Adet %', 'Nakdi Risk %'], [False, True]):
        dft = dfp[[r, c]].copy()
        if fr:
            dft = dft.groupby(r).sum().reset_index().rename(columns={c: n})
            dft[n] /= factor
        else:
            dft = dft.groupby(r).count().reset_index().rename(columns={c: n})
        how = 'inner' if drop_na else 'left'
        df = pd.merge(df, dft, on=r, how=how)
        t = df[n].sum()
        t_list.append(t)
        df[p] = df[n] / t
        t_list.append(1)

        if not fr:
            df = add_v_space(df)
            t_list.append(np.nan)

    if r_dict is not None:
        df[r] = df[r].apply(lambda x: r_dict.get(x, x))

    if sort is not None:
        df = df.sort_values(sort, ascending=False).reset_index(drop=True)

    df = add_h_space(df)
    df.loc[len(df)] = t_list
    df.loc[len(df)] = [f'{last_date} KR202, Reeskontlu, mio TL'] + [np.nan] * (len(df.columns) - 1)
    header = 'Yüksek Riskli Portföy İçerisinde' if high_risk_only else 'Çalışma Kümesi İçerisinde'
    df = insert_header(df, header)

    return insert_corner(df)

### TOP 20

In [171]:
top_column_list = [
    'Müşteri No',
    'Adı',
    'Tahsis Sektör Adı',
    'Nakdi Reeskontlu Toplam',
    'G.Nakdi Toplam',
    'Toplam Reeskontlu Risk',
    'G.Nakdi TL Bakiye',
    'Nakdi TL Reeskontlu',
    'Nakdi Karşılık',
    'Nakdi Karşılık Oranı',
    'Entegre Derece Skor',
    'Yapay Zeka Risk Sınıfı',
    'Gecikme Gün',
    # 'Finansal Güçlük',
    'Bankamız Memzuç Nakdi Risk Payı',
    # 'FG Açıklama',
    # '2024 Öngörüsü',
]

final_top_column_list = [
    'Müşteri No',
    'Adı',
    'Tahsis Sektör Adı',
    'Toplam Reeskontlu Risk',
    'Nakdi Reeskontlu Toplam',
    'G.Nakdi Toplam',
    'Toplam YP Risk Payı',
    'Nakdi Karşılık',
    'Nakdi Karşılık Oranı',
    'Bankamız Memzuç Nakdi Risk Payı',
    'Entegre Derece Skor',
    'Yapay Zeka Risk Sınıfı',
    'Gecikme Gün',
    # 'Finansal Güçlük',
    # 'FG Açıklama',
    '2024 Öngörüsü',
]

renamed_final_top_column_list = [
    'Müşteri No',
    'Firma Adı',
    'Sektör',
    'Toplam Risk',
    'Nakdi Risk',
    'G.Nakdi Risk',
    'Toplam YP Risk Payı',
    'Nakdi Karşılık',
    'Nakdi Karşılık Oranı',
    'Bankamız Nakdi Risk Payı',
    'Entegre Derece / Skor',
    'Yapay Zeka Risk Sınıfı',
    'Gecikme Gün',
    # 'Finansal Güçlük',
    # 'FG Açıklama',
    '2024 Öngörüsü',
]

top_million_columns = [
    'Nakdi Risk',
    'G.Nakdi Risk',
    'Toplam Risk',
    'Nakdi Karşılık',
]

# top_fg_columns = [
#     'Diğer Banka Tahakkuk',
#     'Diğer Banka Takip',
#     'Diğer Banka Tazmin',
#     'Diğer Banka Yapılandırma',
#     'Çek Yasaklılık',
#     'Karşılıksız Çek',
#     'TSPS İflas vb',
# ]

# top_fg_column_desc = [
#     'D. B. Tahakkuk',
#     'D. B. Takip',
#     'D. B. Tazmin',
#     'D. B. Yapılandırma',
#     'Çek Yasaklılık',
#     'Karşılıksız Çek',
#     'TSPS İflas / Konkordato',
# ]

# def get_fg_desc(df):
#     if not df['Finansal Güçlük']: return np.nan
    
#     desc = ''
#     for c, d in zip(top_fg_columns, top_fg_column_desc):
#         if df[c] == 1:
#             desc += d + ', '

#     desc = desc[:-2]
#     desc = desc.replace(', D. B. ', ' & ')
#     desc = desc.replace(' & ', ', ', desc.count('&') - 1)
    
#     return desc

In [172]:
def create_pra_top_risk(df_pra, column, criterion, top=20):
    df = df_pra.copy()
    # df['FG Açıklama'] = df.apply(get_fg_desc, axis=1)
    df = df.loc[(df[column] == criterion) & (df['Risk Kategorisi'].isin(['Çok Yüksek', 'Yüksek'])), top_column_list]
    df = df.fillna(0)

    df['TL Risk'] = df['Nakdi TL Reeskontlu'] + df['G.Nakdi TL Bakiye']
    df['YP Risk'] = df['Toplam Reeskontlu Risk'] - df['TL Risk']
    df['Toplam YP Risk Payı'] = df['YP Risk'] / df['Toplam Reeskontlu Risk']
    yp_sum = df[:top]['YP Risk'].sum()
    total_sum = df[:top]['Toplam Reeskontlu Risk'].sum()
    yp_rest = df[top:]['YP Risk'].sum()
    total_rest = df[top:]['Toplam Reeskontlu Risk'].sum()
    df = pd.merge(df, df_ongoru, on='Müşteri No', how='left')
    df['2024 Öngörüsü'] = df['2024 Öngörüsü'].fillna('-')

    df = df[final_top_column_list]
    df.columns = renamed_final_top_column_list

    df.loc[:, top_million_columns] /= 1_000_000

    df = df.sort_values('Toplam Risk', ascending=False).reset_index(drop=True)
    df['Firma Adı'] = df['Firma Adı'].apply(lambda x: x[:30])
    # df['Finansal Güçlük'] = df['Finansal Güçlük'].map({1: 'VAR', 0: 'YOK'})
    df['Sektör'] = df['Sektör'].map(lambda x: sektor_title_short_dict.get(x, x))

    rest_sums = [f'Diğer {len(df) - top} Firma', np.nan, np.nan]    
    rest_sums += list(df[top:].iloc[:, 3:6].sum())
    rest_sums += [yp_rest/total_rest]
    rest_sums += list(df[top:].iloc[:, 7:8].sum())
    rest_sums += list(df[top:].iloc[:, 8:-1].mean())
    rest_sums += [np.nan]

    sums = ['Genel Toplam', np.nan, np.nan]
    sums += list(df.iloc[:, 3:6].sum())
    sums += [yp_sum/total_sum]
    sums += list(df.iloc[:, 7:8].sum())
    sums += list(df.iloc[:, 8:-1].mean())
    sums += [np.nan]

    df = df[:top]
    df = add_h_space(df)
    df.loc[len(df)] = rest_sums
    df.loc[len(df)] = sums
    df = h_stack([df.iloc[:, :3], df.iloc[:, 3: 7], df.iloc[:, 7: -4], df.iloc[:, -4:]] ,1)
    df.loc[len(df)] = [f'{last_date} KR202, Reeskontlu, mio TL'] + [np.nan] * (len(df.columns) - 1)
    # df['FG Açıklama'] = df['FG Açıklama'].map(lambda x: {0: np.nan}.get(x, x))
    
    return insert_corner(df)

## Firma Kümesi

In [173]:
output_excluded = False

c0 = 'Bankamız Ticari Portföy ({})'.format(last_date)
mn = 'Müşteri No'
nr = 'Nakdi Reeskontlu Toplam'
gnr = 'G.Nakdi Toplam'
tr = 'Toplam Reeskontlu Risk'
c_list = ['Küme', 'Adet', 'Nakdi Risk', 'G.Nakdi Risk', 'Toplam Risk']

df = df_master.copy()
d = 'Çıkarılma Sebebi'
dfd = pd.DataFrame(columns=list(df_master.columns) + [d])
dfk = pd.DataFrame(columns=c_list)

c = c0
dfk.loc[len(dfk)] = return_excluded_info(c, df, False)


if output_excluded:
    c = 'Yakın İzleme / 2. Sınıf'
    dfe = df.loc[(df['Müşteri Sınıfı'] == 2) | (df['İlgili Bölüm'] == 'Krediler İzleme Bölümü')]
    dfe[d] = c
    dfd = pd.concat([dfd, dfe], ignore_index=True)

    c = 'Takip / 3. Sınıf'
    dfe = df.loc[~(df['Müşteri Sınıfı'] <= 2) | (df['İlgili Bölüm'] == 'Ticari ve Kurumsal Krediler Takip Bölümü')]
    dfe[d] = c
    dfd = pd.concat([dfd, dfe], ignore_index=True)
c = 'Standart Nitelikli Ticari Portföy'
pra_bolum_exclude_list = ['Krediler İzleme Bölümü', 'Ticari ve Kurumsal Krediler Takip Bölümü']
df = df.loc[(df['Müşteri Sınıfı'] == 1) & (~df['İlgili Bölüm'].isin(pra_bolum_exclude_list))]
dfk.loc[len(dfk)] = return_excluded_info(c, df, False)


dfk = add_h_space(dfk)

if output_excluded:
    c = 'Banka Risk Grubu'
    dfe = df.loc[(df['Risk Grubu Kodu'] == 'BANKRG')]
    dfe[d] = c
    dfd = pd.concat([dfd, dfe], ignore_index=True)

    c = 'Finans'
    dfe = df.loc[(df['Tahsis Sektör Adı'] == 'FİNANS')]
    dfe[d] = c
    dfd = pd.concat([dfd, dfe], ignore_index=True)

    c = 'PPP (Hastane ve Otoyollar)'
    dfe = df.loc[(df['Tahsis Sektör Adı'] == 'PPP (HASTANE VE OTOYOLLAR)')]
    dfe[d] = c
    dfd = pd.concat([dfd, dfe], ignore_index=True)

    c = 'Belediyeler ve OSBler'
    dfe = df.loc[(df['Tahsis Sektör Adı'] == 'BELEDİYELER VE OSBLER')]
    dfe[d] = c
    dfd = pd.concat([dfd, dfe], ignore_index=True)
c = 'Banka Risk Grubu, Hazine, Finans, PPP & Belediyeler'
pra_sektor_exclude_list = ['FİNANS', 'PPP (HASTANE VE OTOYOLLAR)', 'BELEDİYELER VE OSBLER']
dfe = df.loc[(df['Risk Grubu Kodu'] == 'BANKRG') | (df['Tahsis Sektör Adı'].isin(pra_sektor_exclude_list))]
df = df.loc[(df['Risk Grubu Kodu'] != 'BANKRG') & (~df['Tahsis Sektör Adı'].isin(pra_sektor_exclude_list))]
dfk.loc[len(dfk)] = return_excluded_info(c, dfe)


if output_excluded:
    c = 'Kıbrıs Şubesi'
    dfe = df.loc[(df['Şube Türü'] == 'Kıbrıs')]
    dfe[d] = c
    dfd = pd.concat([dfd, dfe], ignore_index=True)

    c = 'Yurt Dışı Şubesi'
    dfe = df.loc[(df['Şube Türü'] == 'Yurt Dışı')]
    dfe[d] = c
    dfd = pd.concat([dfd, dfe], ignore_index=True)

    c = 'Serbest Bölge Şubesi'
    dfe = df.loc[(df['Şube Türü'] == 'Serbest Bölge')]
    dfe[d] = c
    dfd = pd.concat([dfd, dfe], ignore_index=True)

    c = 'Özel Bankacılık Şubesi'
    dfe = df.loc[(df['Şube Türü'] == 'Özel Bankacılık')]
    dfe[d] = c
    dfd = pd.concat([dfd, dfe], ignore_index=True)
c = 'Yurt Dışı, Kıbrıs & Özel Bankacılık'
pra_sube_exclude_list = ['Kıbrıs', 'Yurt Dışı', 'Özel Bankacılık', 'Serbest Bölge']
dfe = df.loc[df['Şube Türü'].isin(pra_sube_exclude_list)]
df = df.loc[~df['Şube Türü'].isin(pra_sube_exclude_list)]
dfk.loc[len(dfk)] = return_excluded_info(c, dfe)



if output_excluded:
    c = 'Toplam Risk < 1 Milyon TL'
    dfe = df.loc[(df['Toplam Reeskontlu Risk'] < 1_000_000)]
    dfe[d] = c
    dfd = pd.concat([dfd, dfe], ignore_index=True)
    
    c = 'Mali Tablosu Eksik'
    dfe = df.loc[(df['FST_YR'].isnull())]
    dfe[d] = c
    dfd = pd.concat([dfd, dfe], ignore_index=True)
    
    c = 'Son Ay Derecelendirme Eksik'
    dfe = df.loc[(df['Entegre Derece Skor'].isnull())]
    dfe[d] = c
    dfd = pd.concat([dfd, dfe], ignore_index=True)
    
    c = 'Son Ay Risk Skoru Eksik'
    dfe = df.loc[(df['Yapay Zeka Risk Sınıfı'].isnull())]
    dfe[d] = c
    dfd = pd.concat([dfd, dfe], ignore_index=True)
c = 'Toplam Risk < 1 Milyon TL'
dfe = df.loc[
    (df['Toplam Reeskontlu Risk'] < 1_000_000) |
    (df['FST_YR'].isnull()) |
    (df['Entegre Derece Skor'].isnull()) |
    (df['Yapay Zeka Risk Sınıfı'].isnull())
]
df = df.loc[
    (df['Toplam Reeskontlu Risk'] >= 1_000_000) &
    (df['FST_YR'].notnull()) &
    (df['Entegre Derece Skor'].notnull()) &
    (df['Yapay Zeka Risk Sınıfı'].notnull())
]
dfk.loc[len(dfk)] = return_excluded_info(c, dfe)


dfx = dfk.copy().dropna()
neg_sums = [np.sum([min(x, 0) for x in dfx[x]]) for x in dfx.columns[1:]]
dfk.loc[len(dfk)] = ['Toplam Hariç Tutulan Küme'] + neg_sums
dfk = add_h_space(dfk)

c = 'İncelenen Firmalar'
dfe = df.copy()
dfk.loc[len(dfk)] = return_excluded_info(c, dfe, False)
dfk.loc[len(dfk)] = ['İncelenen Firmalar/Ticari Portföy'] + return_percentages('İncelenen Firmalar')[1:]

dfk.loc[len(dfk)] = [f'{last_date} KR202, Reeskontlu, mio TL'] + [np.nan] * (len(dfk.columns) - 1)
kcl = list(dfk.columns)
kcl = kcl[0:1] + [np.nan] + kcl[1:]
dfk[np.nan] = np.nan
dfk = dfk[kcl]

df_pra_kume = insert_corner(dfk)
df_pra_cikarilan = dfd.copy()
df_pra_backup = df.copy()

In [174]:
if output_excluded:
    excluded_column_list = ['Müşteri No', 'Adı', 'Nakdi Reeskontlu Toplam', 'G.Nakdi Toplam', 'Toplam Reeskontlu Risk', 'Çıkarılma Sebebi']
    excluded_renamed_column_list = ['Müşteri No', 'Firma Adı', 'Nakdi Risk', 'G.Nakdi Risk', 'Toplam Risk', 'Çıkarılma Sebebi']
    dfpc = df_pra_cikarilan.copy()
    dfpc = dfpc[excluded_column_list]
    dfpc.columns = excluded_renamed_column_list
    quick_export(dfpc, f'PRA {pra_v} {last_date} Çık', open_file=False)

## Analiz

In [175]:
df_pra, df_pra_yr, df_pra_puan = create_pra_analysis(df_pra_backup)
df_pra_summary = create_pra_summary(df_pra)

In [176]:
new_filtered_bolum_list = [
    'Kurumsal Krediler Tahsis Bölümü',
    'Ticari Krediler Tahsis Bölümü', 
    'PKTB - Bölge Yetkili',
    'PKTB - Şube Yetkili',
    'Proje Finansmanı Bölümü',
]

def fix_bolum(df):
    if df['İlgili Bölüm'] == 'Perakende Krediler Tahsis Bölümü':
        if df['Yetki Kodu'] == 1:
            return 'PKTB - Şube Yetkili'
        else:
            return 'PKTB - Bölge Yetkili'
    else:
        return df['İlgili Bölüm']

df_pra_bf = df_pra.copy()
df_pra_bf['İlgili Bölüm'] = df_pra_bf.apply(fix_bolum, axis=1)

In [177]:

df_pra_segment_dist = create_pra_dist(df_pra, 'Değerlendirmeye Esas Segment', segment_list, rename_dict=segment_title_dict)
df_pra_bolum_dist = create_pra_dist(df_pra_bf, 'İlgili Bölüm', new_filtered_bolum_list)
df_pra_sube_dist = create_pra_dist(df_pra, 'Şube Türü', ['Kurumsal', 'Ticari', 'Karma'])
df_pra_eds_dist = create_pra_dist(df_pra, 'Entegre Derece Skor', [(1, 5), (6, 7), (8, 12)], True)
df_pra_sektor_dist = create_pra_dist(df_pra, 'Tahsis Sektör Adı', sektor_list, False, True, sektor_title_dict)

In [178]:

df_pra_sektor_segment_cross = create_pra_cross_risk(df_pra, 'Tahsis Sektör Adı', 'Değerlendirmeye Esas Segment', sektor_list, segment_list, sektor_title_dict, segment_title_list)
df_pra_sektor_count_risk = create_pra_count_risk(df_pra, 'Tahsis Sektör Adı', sektor_list, sektor_title_dict)

In [179]:
df_pra_kktb_top_risk = create_pra_top_risk(df_pra, 'İlgili Bölüm', 'Kurumsal Krediler Tahsis Bölümü')
df_pra_tktb_top_risk = create_pra_top_risk(df_pra, 'İlgili Bölüm', 'Ticari Krediler Tahsis Bölümü')
df_pra_pktb_top_risk = create_pra_top_risk(df_pra, 'İlgili Bölüm', 'Perakende Krediler Tahsis Bölümü')

In [180]:
# df_pra_sunum = create_pra_sunum(df_pra)

# df_pra_ard_segment = create_count_risk_dist(df_pra, 'Değerlendirmeye Esas Segment')
# df_pra_ard_sektor = create_count_risk_dist(df_pra, 'Tahsis Sektör Adı')
# df_pra_ard_sektor_yr = create_count_risk_dist(df_pra_yr, 'Tahsis Sektör Adı')

# dfpaas = insert_header(df_pra_ard_sektor, 'İncelenen Portföy İçerisinde')
# dfpaasyl = insert_header(df_pra_ard_sektor_yr, 'Yüksek Riskli Portföy İçerisinde')
# df_pra_ard = h_stack([dfpaas, dfpaasyl], 1)

## Korelasyon Matrisi

In [181]:
# # df = load_pickle('df_pra_v12_2212_max')
# import matplotlib.pyplot as plt
# import seaborn as sns

# show_one_half = False
# save_figure = True
# rearrange = True
# correlation_matrix_file_name  ='PRA {} {}.png'.format(pra_v, last_date)

# x = ['Müşteri Sınıfı Max'] + [x for x in df.columns if not x.startswith('Müşteri') and not x.endswith('Puanı')]
# dfcmt = df[x].copy()
# dfcmt = dfcmt.select_dtypes(exclude=['O'])
# correlation_matrix = dfcmt.corr()

# if rearrange:
#     x = list(correlation_matrix.sort_values('Müşteri Sınıfı Max', ascending=False).index)
#     dfcmt = df[x].copy()
#     correlation_matrix = dfcmt.corr()

# upper_half = np.triu(correlation_matrix) if show_one_half else None
# # fig = plt.figure(figsize=(df.shape[1], df.shape[1]), dpi=480)
# # sns.heatmap(correlation_matrix, annot=True, fmt='.2f', square=True, mask=upper_half)

# fig = plt.figure(figsize=(40, 40), dpi = 350)
# sns.heatmap(correlation_matrix, annot=True, fmt='.2f', square=True, mask=upper_half)

# if save_figure:
#     plt.savefig(output_folder_path + correlation_matrix_file_name, bbox_inches = 'tight')
#     plt.close()

In [182]:
# if output_extra:
#     show_one_half = False
#     save_figure = True
#     rearrange = True
#     correlation_matrix_file_name  ='PRA {} {}.png'.format(pra_v, last_date)

#     x = ['Toplam Puan'] + puan_column_list[:-1]
#     x = [y for y in x if y != 'Diğer Banka Tahakkuk Puanı']
#     dfcmt = df_pra[x].copy()
#     dfcmt.columns = x
#     correlation_matrix = dfcmt.corr()

#     if rearrange:
#         x = list(correlation_matrix.sort_values('Toplam Puan', ascending=False).index)
#         dfcmt = df_pra[x].copy()
#         dfcmt.columns = x
#         correlation_matrix = dfcmt.corr()

#     upper_half = np.triu(correlation_matrix) if show_one_half else None
#     # fig = plt.figure(figsize=(df.shape[1], df.shape[1]), dpi=480)
#     # sns.heatmap(correlation_matrix, annot=True, fmt='.2f', square=True, mask=upper_half)

#     fig = plt.figure(figsize=(12, 12), dpi = 350)
#     sns.heatmap(correlation_matrix, annot=True, fmt='.2f', square=True, mask=upper_half)

#     if save_figure:
#         plt.savefig(output_folder_path + correlation_matrix_file_name, bbox_inches = 'tight')
#         plt.close()

# Başlıklar

In [183]:
sheet_dict = {
    'Toplam Puan': df_pra_puan,
    'Küme': df_pra_kume,
    'Özet': df_pra_summary,
    'Segment': df_pra_segment_dist,
    'Bölüm': df_pra_bolum_dist,
    'Şube Türü': df_pra_sube_dist,
    'EDS': df_pra_eds_dist,
    'Sektör': df_pra_sektor_dist,
    'Sektör - Segment': df_pra_sektor_segment_cross,
    'Sektör - Yüksek Riskli': df_pra_sektor_count_risk,
    'KKTB': df_pra_kktb_top_risk,
    'TKTB': df_pra_tktb_top_risk,
    'PKTB': df_pra_pktb_top_risk,
    'Yüksek Riskli Firmalar': df_pra_yr,
}

sheet_dict_list = [sheet_dict]

sheet_list = [item for sublist in [list(x.keys()) for x in sheet_dict_list] for item in sublist]
df_list = [item for sublist in [list(x.values()) for x in sheet_dict_list] for item in sublist]


# sheet_dict_list = [pra_sheet_dict]

# page_list = [item for sublist in [list(x.keys()) for x in sheet_dict_list] for item in sublist]
# df_list = [item for sublist in [list(x.values()) for x in sheet_dict_list] for item in sublist]
# sheet_list = list(range(len(page_list)))

# sheet_list_list = []
# ll = [len(x.keys()) for x in sheet_dict_list]
# c = 0
# for i, l in enumerate(ll):
#     sheet_list_list.append(list(range(c, c + l)))
#     c += l

# pra_sheet_list = sheet_list_list


# df_index = pd.DataFrame()
# sheet_list = [x for x in range(len(page_list))]
# df_index['Sayfa'] = sheet_list
# df_index['Başlık'] = df_index['Sayfa'].apply(lambda x: '=HYPERLINK("#{}!A1", "{}")'.format(str(x), page_list[x]))
# sheet_color_list = ['E5E1E6', 'C7CEEA', 'B2CBF7', 'B5EAD7', 'E2F0CB', 'F7ECC8', 'FFDAC1', 'FFB7B2', 'FF9AA2', 'DCB5CB', 'AF8FC1']

# Final XLSX

In [184]:
output_file_name = 'Panel v64 PRA ' + pra_v + ' ' + last_date
open_file = True

color_long_multi_trend, color_multi_trend, color_trend, row_based = True, True, True, True
output_file_path = output_folder_path + output_file_name + '.xlsx'
writer = pd.ExcelWriter(output_file_path, engine = 'xlsxwriter')

workbook = writer.book
for sheet, df in zip(sheet_list, df_list):
    df.to_excel(writer, sheet_name = sheet, index = False)

    worksheet = writer.sheets[sheet]
    # worksheet.hide_gridlines(2)

writer.close()

if open_file:
    os.startfile(output_file_path)

In [185]:
if save_output:
    save_pickle(df_pra, 'df_pra_{}_{}{}'.format(pra_v, last_date[2:4], last_date[-2:]))

# if output_extra:
#     quick_export(df_pra, 'PRA {} Tüm Firmalar ({})'.format(pra_v, last_date), open_file=False)